In [1]:
!pwd

/truejit/evaluation/smartjit


In [2]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Palatino"],
})

In [3]:
import json
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import time
import subprocess
import pandas as pd
from Profiling import CompilationServiceProfile

In [4]:
from jetstream.gcc_loops.benchmark import gcc_loops
from jetstream.float_mm.benchmark import float_mm
from ffmpeg.benchmark import ffmpeg

# benchmark = ffmpeg
benchmark = gcc_loops

In [ ]:
from mibench.suite import mibench
from spec.suite import spec
from coremark.benchmark import coremark
from ffmpeg.benchmark import ffmpeg
from sqlite.benchmark import sqlite
from jetstream.suite import jetstream
from polybench.suite import suite as polybench
from npb.suite import suite as npb
from wabench.suite import wabench

all_benchmarks = []
# all_benchmarks.extend(jetstream.benchmarks)
# all_benchmarks.extend([coremark])
# all_benchmarks.extend([sqlite])
# all_benchmarks.extend(mibench.benchmarks)
# all_benchmarks.extend(spec.benchmarks)
# all_benchmarks.extend(npb('S').benchmarks)
# all_benchmarks.extend(polybench('standard').benchmarks)
all_benchmarks.extend(wabench.benchmarks)
all_benchmarks.extend([ffmpeg])

In [ ]:
def plot_speedups(benchmark_name, speedups):
    fig, ax = plt.subplots(figsize=(20, 3), dpi=320)
    fs = range(len(speedups))
    y = [speedup for _, speedup in speedups]
    ax.bar(fs, y, color=['red' if speedup < 0 else 'green' for speedup in y])

    avg_speedup = sum(y) / len(y)
    ax.axhline(avg_speedup, color='black', linestyle='--', linewidth=1)

    ax.set_xlabel('Functions')
    ax.set_ylabel('Speedup')

    ax.margins(x=0)

    # format y-axis to percentage and latex friendly
    def format_fn(tick_val, tick_pos):
        val = tick_val * 100
        val = f'{int(val)}' if val.is_integer() else f'{val:.0f}'
        # return with \% sign
        return r'$\mathbf{' + val + r'\%}$'

    ax.yaxis.set_major_formatter(plt.FuncFormatter(format_fn))

    plt.tight_layout()

    speedups_dir = Path('./out/specs-speedups')
    speedups_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(speedups_dir / f'precise-timing.{benchmark_name}.speedups.pdf', bbox_inches='tight', pad_inches=0.1,
                dpi=320,
                format='pdf')

    plt.show()

In [ ]:
def plot(benchmark_name, funcs, unspecialized_profiles, specialized_profiles):
    funcs_unspecialized = []
    funcs_specialized = []
    for f_idx, _ in funcs:
        funcs_unspecialized.append([profile[f_idx] for profile in unspecialized_profiles])
        funcs_specialized.append([profile[f_idx] for profile in specialized_profiles])

    funcs_unspecialized = funcs_unspecialized * 1
    funcs_specialized = funcs_specialized * 1
    funcs = funcs * 1

    def avg(lst):
        return sum(lst) / len(lst)

    speedups = []
    for i, (f_idx, name) in enumerate(funcs):
        avg_unspecialized = avg(funcs_unspecialized[i])
        avg_specialized = avg(funcs_specialized[i])
        speedup = (avg_unspecialized - avg_specialized) / avg_specialized
        speedups.append((i, speedup))

    # sort by speedup
    speedups.sort(key=lambda x: x[1], reverse=True)

    plot_speedups(benchmark_name, speedups)

    num_funcs = len(funcs)
    # num_funcs = 20

    ncols = 8
    nrows = num_funcs // ncols
    nrows += 1 if num_funcs % ncols else 0

    fig, axs = plt.subplots(figsize=(25, 1.1 * nrows), nrows=nrows, ncols=ncols, dpi=320)
    cnt = 0
    for i, speedup in speedups:
        if cnt >= num_funcs:
            break
        ax = axs.flatten()[cnt]
        cnt += 1

        specs = ax.violinplot(funcs_specialized[i], showmeans=False, showmedians=False, showextrema=False, vert=False)
        for pc in specs['bodies']:
            pc.set_facecolor('blue')
            pc.set_edgecolor('black')
            pc.set_linewidth(.25)
            pc.set_alpha(.5)
        unspecs = ax.violinplot(funcs_unspecialized[i], showmeans=False, showmedians=False, showextrema=False,
                                vert=False)
        for pc in unspecs['bodies']:
            pc.set_facecolor('red')
            pc.set_edgecolor('black')
            pc.set_linewidth(.25)
            pc.set_alpha(.5)

        avg_specialized = avg(funcs_specialized[i])
        avg_unspecialized = avg(funcs_unspecialized[i])
        title = f'{avg_unspecialized / 1000000:>.1f}' r' $\Longrightarrow$ ' f'{avg_specialized / 1000000:>.1f} ms'
        title += f' | speedup {speedup * 100:.0f}' r'\%'
        speedup_color = 'green' if speedup >= 0 else 'red'
        ax.set_title(title, loc='center', fontweight='bold', color=speedup_color)

        # format x-axis to ms
        def format_fn(tick_val, tick_pos):
            val = tick_val / 1_000_000
            return f'{int(val)}' if val.is_integer() else f'{val:.2f}'

        ax.xaxis.set_major_formatter(plt.FuncFormatter(format_fn))
        ax.set_xlabel('Time (ms)')
        ax.set_yticks([])

    for i in range(cnt, nrows * ncols):
        fig.delaxes(axs[i // ncols, i % ncols])

    plt.tight_layout()

    # to pdf
    speedups_violin_dir = Path('./out/specs')
    speedups_violin_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(speedups_violin_dir / f'precise-timing.{benchmark_name}.pdf', bbox_inches='tight', pad_inches=0.1,
                dpi=320,
                format='pdf')

    plt.show()

In [ ]:
from mibench.suite import mibench
from spec.suite import spec
from coremark.benchmark import coremark
from ffmpeg.benchmark import ffmpeg
from sqlite.benchmark import sqlite
from jetstream.suite import jetstream
from polybench.suite import suite as polybench
from npb.suite import suite as npb
from wabench.suite import wabench

all_benchmarks = []
all_benchmarks.extend(jetstream.benchmarks)
all_benchmarks.extend([coremark])
all_benchmarks.extend([sqlite])
all_benchmarks.extend(mibench.benchmarks)
all_benchmarks.extend(spec.benchmarks)
all_benchmarks.extend(npb('S').benchmarks)
all_benchmarks.extend(polybench('mini').benchmarks)
all_benchmarks.extend(wabench.benchmarks)
all_benchmarks.extend([ffmpeg])

In [ ]:
reps = 10
for benchmark in all_benchmarks:
    print(f'benchmark={benchmark.mode}')
    funcs = profile_for_specialization(benchmark)
    print(f'specializing {len(funcs)} functions')
    unspecialized_profiles, specialized_profiles = profile_n_times(benchmark, reps)
    plot(benchmark.mode, funcs, unspecialized_profiles, specialized_profiles)